In [1]:
import os
from pathlib import Path
import sys

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR100
from torchvision.models import resnet18
from torchvision.transforms import v2

act_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
sys.path.insert(0, act_root) if act_root not in sys.path else None

ACT_ROOT = Path(act_root)

BATCH_SIZE = 1
EPOCHS = 5

MODEL_SAVE_PATH = ACT_ROOT / "ipynb/resnet18_cifar100.pth"

# Train model on `float32` dataset
model = resnet18()

train_dataset = CIFAR100(root=ACT_ROOT / "data/torchvision/CIFAR100/raw", transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]))
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
size = len(train_dataloader.dataset)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters())

if os.path.exists(MODEL_SAVE_PATH):
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, weights_only=True))
else:
    model.train()

    for _ in range(EPOCHS):
        for batch, (X, y) in enumerate(train_dataloader):
            # Compute prediction and loss
            pred = model(X)
            loss = loss_fn(pred, y)

            # Backpropagation
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            if batch % 100 == 0:
                loss, current = loss.item(), batch * BATCH_SIZE + len(X)
                print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

    torch.save(model.state_dict(), MODEL_SAVE_PATH)

In [2]:
from act.front_end.model_synthesis import synthesize_models_from_specs
from act.front_end.torchvision_loader.create_specs import TorchVisionSpecCreator
from act.front_end.vnnlib_loader.create_specs import VNNLibSpecCreator


USE_VNNLIB = False

# Without specifying `dtype = float32`, both of these spec creators
# should fail - the default data type for frontends is `float64`
if USE_VNNLIB:
    spec_creator = VNNLibSpecCreator()
    specs = spec_creator.create_specs_for_data_model_pairs(
        categories=["cifar100_2024"],
        max_instances=10
    )
else:
    spec_creator = TorchVisionSpecCreator()
    specs = []

    for i in range(10):
        spec = spec_creator._create_specs_for_single_instance(
            data_source="CIFAR100",
            model_name="resnet18",
            pytorch_model=model,
            dataloader=train_dataloader,
            num_samples=1,
            start_index=0,
            validate_shapes=True
        )
        specs.append(spec)

synthesis_models = synthesize_models_from_specs(specs)

[ACT] Auto-detecting project root: ..
[ACT] Gurobi license found: ../modules/gurobi/gurobi.lic

🧬 Synthesizing models from 10 spec result(s)...

🎉 Synthesis Complete:
   Total specs: 240
   Wrapped models: 4


In [8]:
from act.front_end.spec_creator_base import LabeledInputTensor
from act.front_end.verifiable_model import InputLayer, VerifiableModel
from act.pipeline.fuzzing.actfuzzer import ACTFuzzer, FuzzingConfig


config_path = Path(ACT_ROOT) / "act/pipeline/fuzzing/config.yaml"
config = FuzzingConfig.from_yaml(
    config_path=config_path,
    max_iterations=500,
    timeout_seconds=60.0,
    report_interval=20,
    verbose=1,
    # Tracing: capture execution traces for analysis
    trace_level=1,
    trace_storage='json'
)

fuzzers = []
reports = []

for key, model in synthesis_models.items():
    verif_model: VerifiableModel = model
    
    input_layer: InputLayer = verif_model.input_layer
    labeled_input = input_layer.labeled_input

    images = labeled_input.tensor.to("cpu")
    labels = labeled_input.label.to("cpu")
    B = images.shape[0]

    initial_seeds = [
        LabeledInputTensor(tensor=images[i:i+1], label=labels[i:i+1])
        for i in range(B)
    ]
    
    fuzzer = ACTFuzzer(
        wrapped_model=model,
        initial_seeds=initial_seeds,
        config=config
    )
    
    report = fuzzer.fuzz()

    fuzzers.append(fuzzer)
    reports.append(report)

[MutationEngine] Adaptive Scalar Perturbation Size:
  - perturb_scale: 0.1 (fraction of range per perturbation)
  - mean_range: 0.093250
  - computed perturb_size: 0.009325
  - steps_to_traverse: ~10.0 steps
  - interpretation: Each mutation perturbation covers 10.0% of the range
📊 Tracing enabled: Level 1, sampling every 1 iteration(s)
   Output: ../act/pipeline/log/fuzzing_results/traces_0.json
ACT: Abstract Constraint Transformer
Inference-based whitebox fuzzing for neural network verification

🚀 Starting ACTFuzzer with 10 seeds
   Device: cpu
   Batch size: 80 (from model synthesis)
   Max iterations: 500
   Timeout: 60.0s

📊 Iteration     80 | GlobalCov: 94.00% BestInputCov: 86.15% | Seeds:   90 | Violations:  74 (+74) | Speed:  30.0 it/s (2403 samples/s)
📊 Iteration    160 | GlobalCov: 94.67% BestInputCov: 86.24% | Seeds:  170 | Violations: 154 (+80) | Speed:  25.8 it/s (2064 samples/s)
📊 Iteration    240 | GlobalCov: 95.25% BestInputCov: 86.24% | Seeds:  250 | Violations: 234 (+